# SA-CUT — E1 Colab training (Ch5 main experiment)

Trains the **full SA-CUT model** (E1 in the Ch5 experiment matrix): the reference run
that all baselines (E2) and ablations (E3) are compared against.

**Fresh run:** Cell 1 → 2 → 3 → 4 (smoke) → 5A (short, 20 epochs) → 5B (full).
5A is not optional boilerplate — it is where you confirm `loss_D` is healthy before
committing hours of A100 time.

**Continuing the existing E1 run:** Cell 1 → 2 → 3 → **STEP 6** (resume with a rebalanced
discriminator) → **STEP 7** (grade the result). See STEP 6 for why the resume changes two
hyper-parameters.

**Why this notebook instead of `SA_CUT_Colab_Train.ipynb`:** that notebook builds its
experiment YAML from a Python dict listing only four loss keys, so anything not in that
list silently falls back to `configs/default.yaml`. Since `default.yaml` keeps
`color_loss_mode: global` (so ablation baselines stay unchanged), that path would train
**without the region-conditioned colour loss**. This notebook uses
`configs/experiment_sa_cut_full.yaml` directly and only overrides paths on the CLI, so
every component stays exactly as committed.

**Design (same three rules as the SQ-MIL bootstrap):**
1. **Code on Colab local disk**, pulled from GitHub — fast, always the committed config.
2. **Data copied Drive → local disk once per session** — training reads many small patch
   files per epoch; the Drive FUSE mount is far slower than local disk.
3. **Checkpoints written back to Drive** — Colab disconnects; weights must survive.

**Watch while training (the mandatory training rule):** `loss_D` must stay in
**0.3–0.7**. If it falls below 0.1 the adversarial gradient to G vanishes and G degrades
to "colourised TPAF" instead of H&E. The first E1 run drifted to `d_loss_ema = 0.129`,
which is what STEP 6 corrects.

Expected on a healthy run: `struct=0.0000` for the first 3 epochs (warm-up) then ramping
to `lambda_struct=5.0` over 5 epochs; `color` noticeably larger than in `global` mode.

In [ ]:
# ── Cell 1: mount Drive + config ─────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

# --- edit here if your paths differ ---
os.environ['DRIVE']    = '/content/drive/MyDrive/SA-CUT'
os.environ['REPO_URL'] = 'https://github.com/z-pan/SA-CUT.git'
os.environ['BRANCH']   = 'main'
os.environ['RUN_NAME']  = 'E1_sa_cut_full'

assert os.path.isdir(os.environ['DRIVE']), f"Drive folder not found: {os.environ['DRIVE']}"
print('GPU:'); os.system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')

In [ ]:
# ── Cell 2: code → local disk (clone, or pull if present) + deps ──
%cd /content
!if [ -d SA-CUT ]; then cd SA-CUT && git fetch && git checkout "$BRANCH" && git pull --ff-only; else git clone --branch "$BRANCH" "$REPO_URL"; fi
%cd /content/SA-CUT
!pip install -q tifffile pytorch-fid pyyaml scikit-image
# Confirm the region-conditioned colour loss commit is present.
!git log --oneline -3
!test -f losses/region_color_loss.py && echo 'OK: region_color_loss.py present'

In [ ]:
# ── Cell 3: data Drive → local disk (once per session) ──
# rsync --ignore-existing makes reconnects near-instant.
!mkdir -p data/raw/tpaf data/raw/he data/patches/masks
!rsync -a --ignore-existing "$DRIVE/patches/tpaf/"  data/raw/tpaf/
!rsync -a --ignore-existing "$DRIVE/patches/he/"    data/raw/he/
!rsync -a --ignore-existing "$DRIVE/patches/masks/" data/patches/masks/

# Masks are matched to TPAF patches by filename stem, so a mask must exist for
# every TPAF patch in precomputed mode.
!echo "tpaf=$(ls data/raw/tpaf | wc -l)  he=$(ls data/raw/he | wc -l)  masks=$(ls data/patches/masks | wc -l)"

In [ ]:
# ── Cell 4: smoke test first (~30 s) ──
# Same check as locally: CPU, 1 epoch, tiny synthetic data. Verifies the Colab
# environment and the committed code before spending A100 hours.
!bash scripts/smoke_test.sh --config configs/experiment_sa_cut_full.yaml

## STEP 5A — short validation run (RUN THIS FIRST)

**Do not skip ahead to the full run.** 20 epochs at constant LR, ~1/20th of the cost, to
confirm the run is healthy before committing hours of A100 time.

Check in the epoch log:

| Field | Healthy | Bad |
|---|---|---|
| `D=` | **0.3–0.7** | `< 0.1` → D collapse; G degrades to colourised TPAF |
| `D_gate=` | low % | pegged near 100% → D permanently gated |
| `struct=` | `0.0000` for epochs 0–2, then rising | still 0 after epoch 5 |
| `color=` | clearly non-trivial (region mode) | ~0 |
| `G=` | fluctuating, no NaN / blow-up | NaN or monotonic explosion |

Also open a sample image under `$DRIVE/results/logs/${RUN_NAME}_short/` — nuclei should be
trending purple, not blank/white (blank nuclei is the UTOM failure mode SA-CUT exists to fix).

Only if all of the above look right, continue to STEP 5B.

In [ ]:
# ── STEP 5A: short validation run — 20 epochs, no LR decay ──
# Separate RUN_NAME suffix so this never overwrites the real E1 checkpoints/logs.
!python scripts/train.py \
    --config configs/experiment_sa_cut_full.yaml \
    --training.n_epochs=20 \
    --training.n_epochs_decay=0 \
    --data.patch_size=512 \
    --data.num_workers=2 \
    --experiment.name="${RUN_NAME}_short" \
    --experiment.checkpoint_dir="$DRIVE/checkpoints" \
    --experiment.log_dir="$DRIVE/results/logs" \
    --experiment.use_wandb=false

## STEP 5B — full E1 run (only after 5A looks healthy)

400 epochs (200 at full LR + 200 linear decay) — the reference run for the Ch5 experiment
matrix. Expect at least one Colab disconnect; checkpoints go to Drive every 10 epochs, so
use the resume cell below.

The first 200 epochs run at constant LR, so **STEP 5A's epochs are numerically identical to
the opening of this run** (same seed, same LR). 5A therefore costs nothing in information
terms — it just lets you bail out early instead of hours in.

In [ ]:
# ── STEP 5B: full E1 training (400 epochs) ──
# Config is used as committed (mask input + SA-PatchNCE + L_struct + region colour
# loss + the D-collapse guards). Only paths and patch size are overridden.
# patch_size=512 matches the 512x512 precomputed masks on Drive.
!python scripts/train.py \
    --config configs/experiment_sa_cut_full.yaml \
    --data.patch_size=512 \
    --data.num_workers=2 \
    --experiment.name="$RUN_NAME" \
    --experiment.checkpoint_dir="$DRIVE/checkpoints" \
    --experiment.log_dir="$DRIVE/results/logs" \
    --experiment.use_wandb=false

## STEP 6 — resume with a rebalanced discriminator

**Read this before resuming.** The first E1 run stopped at **epoch 186/400** with
`d_loss_ema = 0.129` — well below the mandatory 0.3–0.7 band. D had won, so G was getting
a weak/uninformative adversarial signal. Measured on 40 tissue-rich test patches, the
epoch-186 generator gives:

| | epoch 186 | UTOM nuc_hi | real H&E |
|---|---|---|---|
| cytoplasm R−B gap | **−0.1** (solved) | −10.9 (too violet) | 0 |
| nuclei R−B gap | +16.4 (too weak) | +8.4 | 0 |
| **green-dominant pixels** | **13.61 %** | 0.51 % | 0.01 % |

So the region-conditioned colour loss did fix the violet cytoplasm, but the generator is
also painting colours that cannot exist in H&E (13.6 % green) and under-staining nuclei —
both consistent with an over-strong D that G is gaming rather than matching.

**What changed below, and why:**

* `d_loss_gate_threshold` 0.1 → **0.3** — the direct lever. D updates are skipped while
  `EMA(loss_D) < 0.3`, so D is held back until G catches up, then gating disengages by
  itself. Self-regulating, which is why it is the primary change.
* `lr_D` 1e-4 → **5e-5** — makes D adapt more slowly from here on, so it does not simply
  re-win between gated steps.

Everything else is left exactly as trained. Watch `D=` climb into 0.3–0.7 over the first
few epochs; `D_gate=` should start high and fall as G recovers. If green pixels are still
above ~2 % after ~20 more epochs, the next lever is an explicit colour constraint rather
than more D tuning.

In [ ]:
# -- STEP 6: resume E1 from the last checkpoint, with D held back --
# Re-run Cells 1-3 first after a disconnect (Cell 3 is fast on reconnect).
# The two overrides fix the over-strong discriminator; everything else is as trained.
!python scripts/train.py \
    --config configs/experiment_sa_cut_full.yaml \
    --resume "$DRIVE/checkpoints/$RUN_NAME/latest.pth" \
    --training.d_loss_gate_threshold=0.3 \
    --training.lr_D=5e-5 \
    --data.patch_size=512 \
    --data.num_workers=2 \
    --experiment.name="$RUN_NAME" \
    --experiment.checkpoint_dir="$DRIVE/checkpoints" \
    --experiment.log_dir="$DRIVE/results/logs" \
    --experiment.use_wandb=false

## STEP 7 — grade the checkpoint objectively

Run this after (or during) training to decide whether the rebalance worked, instead of
judging by eye. It translates a fixed set of TPAF patches and scores the result against
real H&E.

Targets to beat, from the epoch-186 baseline:

| metric | epoch 186 | goal |
|---|---|---|
| cytoplasm R−B gap | −0.1 | keep near 0 |
| nuclei R−B gap | +16.4 | → 0 (more violet); UTOM reaches +8.4 |
| green-dominant pixels | 13.61 % | **< 1 %** |

The green figure is the one that matters most right now — a run can score a perfect
cytoplasm hue while painting impossible colours, which is exactly what epoch 186 did.

**If the log prints an unexpected epoch**, the checkpoint path is wrong. The trainer
appends `experiment.name` to `checkpoint_dir`, so weights end up in
`$DRIVE/checkpoints/$RUN_NAME/`. Passing `checkpoints/$RUN_NAME` as `checkpoint_dir`
nests them one level deeper and the old file is read instead — check the epoch in the
first log line before trusting the scores.

In [ ]:
# -- STEP 7: translate a held-out set, then score it --
# The trainer appends experiment.name to checkpoint_dir, so the weights live in
# $DRIVE/checkpoints/$RUN_NAME/ -- pass the parent, not the run folder, above.
!python scripts/test.py     --checkpoint "$DRIVE/checkpoints/$RUN_NAME/latest.pth"     --input_dir  data/raw/tpaf     --mask_dir   data/patches/masks     --output_dir results/eval_$RUN_NAME

# Real H&E is the reference domain; masks define nuclei on the generated side.
!python scripts/eval_stain_color.py     --virtual  results/eval_$RUN_NAME     --nuc-mask data/patches/masks     --real     data/raw/he

## Next: E2 baselines and E3 ablations

Same pattern — swap the config, keep the path overrides and `RUN_NAME`. The ablation
configs already exist in `configs/`, so no dict-built YAML is needed:

| Experiment | Config |
|---|---|
| E2 CUT baseline | `configs/ablation_cut_baseline.yaml` |
| E2 CycleGAN | `configs/ablation_cyclegan.yaml` |
| E3 mask input only | `configs/ablation_cut_mask_input.yaml` |
| E3 no `L_struct` | `configs/ablation_sa_cut_no_struct.yaml` |

Give each run its own `RUN_NAME` so checkpoints and logs stay separate on Drive.